# Checkpoints


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

source_str = "orders"
sink_str = "orders_aggregated"

tn = (
    Tn.source(source_str)
    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "product_ids": sorted(agg_r["product_ids"] + [r["product_id"]])},
                  {"orders": 0, "product_ids": []},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "product_ids": agg_r["product_ids"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r})
    .sink(sink_str)
)

built_tn = Tn.build(tn)



In [2]:
import random

class OrderGenerator:
    def __init__(self):
        self.order_id_int = 0
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = 1

    def generate(self):
        m = {
            "key": self.order_id_int,
            "value": {"id": self.order_id_int,
                      "product_id": random.randint(0, 100 - 1),
                      "customer_id": random.randint(0, 10 - 1),
                      "ts": self.ts_int},
        }
        #
        self.order_id_int += 1
        #
        self.ts_int += self.ts_step_int
        #
        return m

#

gen = OrderGenerator()
for _ in range(3):
    print(gen.generate())


{'key': 0, 'value': {'id': 0, 'product_id': 67, 'customer_id': 9, 'ts': 0}}
{'key': 1, 'value': {'id': 1, 'product_id': 89, 'customer_id': 0, 'ts': 1}}
{'key': 2, 'value': {'id': 2, 'product_id': 11, 'customer_id': 0, 'ts': 2}}


In [ ]:
built_tn.reset()
gen = OrderGenerator()
source_m_list = []
sink_m_list = []
for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

source_key_int_value_dict_dict = {}
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["key"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


In [ ]:
import cloudpickle

#

gen = OrderGenerator()


#

built_tn.reset()
source_m_list = []
sink_m_list = []
for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

x = cloudpickle.dumps(built_tn._evaluator)

# print(built_tn.latest())

built_tn.reset()

# print(built_tn.latest())

y = cloudpickle.loads(x)

built_tn._evaluator = y

# print(built_tn.latest())

#

for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

source_key_int_value_dict_dict = {}
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["key"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


In [3]:
import sys
sys.path.insert(1, "..")

import kafi.streams.streams
import importlib
importlib.reload(kafi.streams.streams)

from kafi.kafka.cluster.cluster import Cluster
from kafi.streams.streams import Streams

c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

source_str = "orders"
sink_str = "orders_aggregated"

tn = (
    Streams.source(c, source_str)
    
    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "order_ids": sorted(agg_r["order_ids"] + [r["id"]]),
                                    "product_ids": sorted(agg_r["product_ids"] + [r["product_id"]])},
                  {"orders": 0, "order_ids": [], "product_ids": []},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "order_ids": agg_r["order_ids"],
                                     "product_ids": agg_r["product_ids"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r}).peek("sink")
    .sink(c, sink_str)
)

built_tn = Streams.build(tn)



In [ ]:
from kafi.helpers import get_millis

orders_int = 1000

built_tn.reset()

checkpoint_str = "checkpoint"
g = f"group_{get_millis()}"

c.recreate(source_str)
c.recreate(sink_str)
c.recreate(checkpoint_str)

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, checkpoint_interval=0.01, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)
gen = OrderGenerator()

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



'orders'

Source consumer group offsets for topic orders: {}
(['orders'], 'group_1785129624066')
Checkpoint consumer group offsets for topic checkpoint: {}
(['checkpoint'], 'group_1785129624066_checkpoint')
sink: {'key': 4, 'value': {'customer_id': 4, 'orders': 106, 'order_ids': [0, 8, 16, 63, 67, 70, 89, 97, 128, 130, 136, 141, 143, 165, 171, 178, 182, 193, 200, 217, 224, 251, 254, 262, 288, 293, 303, 304, 312, 314, 323, 327, 332, 347, 409, 416, 420, 428, 452, 453, 467, 476, 481, 489, 492, 503, 506, 509, 519, 527, 531, 532, 541, 542, 545, 550, 563, 570, 588, 597, 603, 605, 618, 620, 646, 669, 680, 684, 694, 719, 734, 751, 755, 758, 764, 770, 771, 777, 786, 793, 802, 809, 811, 822, 835, 839, 854, 863, 870, 873, 877, 887, 892, 894, 895, 897, 899, 932, 935, 936, 941, 956, 985, 986, 993, 998], 'product_ids': [0, 0, 0, 1, 1, 1, 2, 2, 4, 4, 4, 4, 9, 10, 11, 12, 12, 14, 16, 16, 17, 19, 19, 19, 21, 21, 22, 24, 24, 25, 26, 28, 29, 29, 29, 30, 30, 31, 32, 32, 35, 36, 37, 37, 39, 39, 41, 41, 42, 42, 44, 4

In [5]:
await stop_fun()
await Streams.tasks()

Safely stopping Streams...
...done.


[]

In [ ]:
built_tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



{'orders': 1000}
{'orders_aggregated': 10}
{'checkpoint': 31}


'orders'

Source consumer group offsets for topic orders: {0: 999}
(['orders'], 'group_1785129624066')
Checkpoint consumer group offsets for topic checkpoint: {}
(['checkpoint'], 'group_1785129624066_checkpoint')
Loading checkpoint...
...loading checkpoint done.
sink: {'key': 6, 'value': {'customer_id': 6, 'orders': 179, 'order_ids': [4, 12, 24, 33, 46, 50, 53, 55, 58, 64, 69, 73, 77, 114, 119, 126, 137, 139, 149, 166, 169, 186, 194, 206, 209, 230, 233, 244, 275, 282, 291, 294, 302, 315, 317, 319, 329, 331, 346, 364, 372, 425, 437, 439, 449, 472, 515, 547, 557, 558, 574, 581, 582, 584, 585, 606, 634, 675, 677, 697, 713, 717, 721, 744, 745, 749, 760, 797, 805, 812, 828, 850, 855, 859, 884, 916, 917, 925, 930, 931, 946, 947, 972, 973, 999, 1009, 1010, 1022, 1025, 1036, 1052, 1057, 1065, 1066, 1090, 1092, 1101, 1102, 1109, 1113, 1115, 1120, 1128, 1131, 1151, 1153, 1157, 1200, 1204, 1214, 1242, 1269, 1270, 1283, 1304, 1307, 1317, 1333, 1360, 1372, 1413, 1420, 1427, 1429, 1445, 1449, 1454, 1469, 1471

In [7]:
await stop_fun()
await Streams.tasks()

Safely stopping Streams...
...done.


[]

In [ ]:
built_tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)

#

for _ in range(orders_int):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()


{'orders': 2000}
{'orders_aggregated': 31}
{'checkpoint': 227}


'orders'

Source consumer group offsets for topic orders: {0: 1999}
(['orders'], 'group_1785129624066')
Checkpoint consumer group offsets for topic checkpoint: {0: 118}
(['checkpoint'], 'group_1785129624066_checkpoint')
Loading checkpoint...
...loading checkpoint done.
sink: {'key': 4, 'value': {'customer_id': 4, 'orders': 286, 'order_ids': [0, 8, 16, 63, 67, 70, 89, 97, 128, 130, 136, 141, 143, 165, 171, 178, 182, 193, 200, 217, 224, 251, 254, 262, 288, 293, 303, 304, 312, 314, 323, 327, 332, 347, 409, 416, 420, 428, 452, 453, 467, 476, 481, 489, 492, 503, 506, 509, 519, 527, 531, 532, 541, 542, 545, 550, 563, 570, 588, 597, 603, 605, 618, 620, 646, 669, 680, 684, 694, 719, 734, 751, 755, 758, 764, 770, 771, 777, 786, 793, 802, 809, 811, 822, 835, 839, 854, 863, 870, 873, 877, 887, 892, 894, 895, 897, 899, 932, 935, 936, 941, 956, 985, 986, 993, 998, 1027, 1060, 1063, 1069, 1106, 1119, 1123, 1147, 1171, 1184, 1199, 1201, 1205, 1208, 1226, 1240, 1243, 1254, 1256, 1265, 1271, 1280, 1295, 1297, 13

In [9]:
await stop_fun()
await Streams.tasks()

Safely stopping Streams...
...done.


[]

In [10]:
source_key_int_value_dict_dict = {}
source_m_list = c.cat(source_str)
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_order_id_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("order_ids", [])
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "order_ids": sorted(agg_order_id_int_list + [order_id_int]),
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
sink_m_list = c.cat(sink_str)
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["value"]["customer_id"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


(['orders'], '1785129685895')
Read: 1000
Read: 2000
Read: 3000
(['orders_aggregated'], '1785129691150')
{4: {'customer_id': 4, 'orders': 287, 'order_ids': [0, 8, 16, 63, 67, 70, 89, 97, 128, 130, 136, 141, 143, 165, 171, 178, 182, 193, 200, 217, 224, 251, 254, 262, 288, 293, 303, 304, 312, 314, 323, 327, 332, 347, 409, 416, 420, 428, 452, 453, 467, 476, 481, 489, 492, 503, 506, 509, 519, 527, 531, 532, 541, 542, 545, 550, 563, 570, 588, 597, 603, 605, 618, 620, 646, 669, 680, 684, 694, 719, 734, 751, 755, 758, 764, 770, 771, 777, 786, 793, 802, 809, 811, 822, 835, 839, 854, 863, 870, 873, 877, 887, 892, 894, 895, 897, 899, 932, 935, 936, 941, 956, 985, 986, 993, 998, 1027, 1060, 1063, 1069, 1106, 1119, 1123, 1147, 1171, 1184, 1199, 1201, 1205, 1208, 1226, 1240, 1243, 1254, 1256, 1265, 1271, 1280, 1295, 1297, 1318, 1319, 1320, 1328, 1346, 1351, 1352, 1355, 1375, 1380, 1381, 1401, 1406, 1425, 1438, 1461, 1463, 1479, 1483, 1495, 1496, 1506, 1507, 1536, 1539, 1564, 1568, 1577, 1578, 1597, 